## Experiment: Linear Regression over ill-conditioned data
Syntetic data are used to illustrate the behavior of linear regression when applied to ill-conditioned data. The dataset consists of a set of input features and corresponding target values, where the features are highly correlated, leading to multicollinearity issues.

In [1]:
import numpy as np

# Configuration
NOISE = 2
EPOCHS = 20
TEST_STEP = 5

In [2]:
# Function to estimate
def f(x: np.ndarray) -> np.ndarray:
    return x*2 + 2

In [10]:
# Dataset
from core.dataset.dataset import TensorDataset
from core.tensor import Tensor

x = np.linspace(-5, 5, 100)
y = x+1e-6
X = np.stack([x, y]).T
y = f(X)
dataset_tr, dataset_te = TensorDataset(Tensor(X), Tensor(y)).split()
X_tr, y_tr = dataset_tr
X_te, y_te = dataset_te

print(f'X.shape: {X.shape}, y.shape: {y.shape}')
print(f'X_tr.shape: {X_tr.shape}, y_tr.shape: {y_tr.shape}')
print(f'X_te.shape: {X_te.shape}, y_te.shape: {y_te.shape}')
print(f'condition number: k(y) = {np.linalg.cond(y)}')
print(f'condition number: k(y@y) = k(y)^2 = {np.linalg.cond(y.T@y)}')


X.shape: (100, 2), y.shape: (100, 2)
X_tr.shape: (90, 2), y_tr.shape: (90, 2)
X_te.shape: (10, 2), y_te.shape: (10, 2)
condition number: k(y) = 6517456.403997613
condition number: k(y@y) = k(y)^2 = 42316803842755.09


## Dataset 
The dataset is generated using a linear model with added noise. The input features are created to be highly correlated, which can cause instability in the estimation of regression coefficients. The target variable is generated as a linear combination of the input features plus some random noise.

The dataset is split into training and testing sets to evaluate the performance of the linear regression model.
The training set is 95% of data meanwhile the test set is 5%

In [7]:
from core.optimizer import SGD
from core.losses import MSELoss
from core.layers import Linear, Sequential

# Linear Regression
model = Sequential(
    Linear(2, 2)
)
loss = MSELoss()
optimizer = SGD(model.parameters, 1e-4)

In [8]:
# Training loop
assert(EPOCHS % TEST_STEP == 0)
train_losses:list[float] = []
test_losses:list[float] = []

for i in range(EPOCHS):
    pred = model(X_tr)
    train_loss = loss(pred, y_tr)
    train_losses.append(train_loss)
    if i % TEST_STEP == 0 or i == EPOCHS-1:
        model.eval()
        pred_test = model(X_te)
        test_loss = loss(pred_test, y_te)
        test_losses.append(test_loss)
        model.train()

    train_loss.backward()
    optimizer.step()
    

In [ ]:
import matplotlib.pyplot as plt

out = model(X_te)

# Plot in one window, divided into sections
fig, (ax_loss, ax_result) = plt.subplots(1, 2, figsize=(12, 5))

# Loss section
ax_loss.plot(list(range(EPOCHS)), train_losses, label="train")
ax_loss.plot(list(range(0, EPOCHS+1, TEST_STEP)), test_losses, label="test")
ax_loss.set_title("Loss")
ax_loss.set_xlabel("epoch")
ax_loss.set_ylabel("loss")
ax_loss.legend()

# Result section (the regression)
ax_result.plot(X_te.data, y_te.data, 'r.', label="target")
ax_result.plot(X_te.data, out.data, label="prediction")
ax_result.plot(X_te.data, y_test, label='to estimate')
ax_result.plot(X_te.data, closed_form, label='closed form')
ax_result.set_title("Regression")
ax_result.set_xlabel("x")
ax_result.set_ylabel("y")
ax_result.legend()

fig.tight_layout()
plt.show()